# B2-024-gpu-scientific-ml-capstone — Practice p25 — Solution

**Type:** scenario · **Difficulty:** core · **Concepts:** semi-supervised-pseudo-labeling

*55 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Solution

### (a) Diagnosis

Two linked failure modes: **confirmation bias** (the model re-admits and reinforces its own mistakes with growing confidence) and the resulting **class collapse / imbalance amplification** (the pseudo-labels drift toward classes 0 and 1 and class 2 is starved out). Evidence from five columns:

* *Pseudo-labels admitted:* class 0 grows $180\to260\to330\to352$ while class 2 shrinks $90\to40\to12\to2$. The pool is roughly balanced (about 200 per class), so 352 class-0 pseudo-labels in round 4 means **at least about 150 of them are wrong**, almost certainly class-2 (and some class-1) images labelled 0.
* *Validation recall:* class 2 falls $0.80\to0.80\to0.60\to0.30\to0.20$ while class 0 rises to $1.00$: the model increasingly predicts class 0 for everything, which is exactly what the admitted counts predicted.
* *Validation accuracy:* peaks at $0.83$ in round 1, then falls to $0.80$, $0.73$, $0.70$, below the labelled-only round 0 ($0.80$).
* *Mean confidence of admitted:* rises $0.91\to0.95\to0.98\to0.99$ while accuracy falls.
* *Training loss:* falls $0.004\to0.001$.

**Why falling loss and rising confidence are not evidence of improvement.** The training loss is measured on the model's own pseudo-labels, so it only shows that the student fits the labels it was given, including the wrong ones; a model that labels everything class 0 and then trains on that has a tiny loss. Confidence is the model's self-assessment: a student trained (without noise, from the teacher's weights) to reproduce the teacher's labels becomes *more* confident about the same decisions, right or wrong. Neither is measured against true labels; only the labelled validation columns are, and they get worse.

### (b) Mechanism

Continuing from the teacher's weights means the student starts exactly at the teacher's decision boundary, and training it, without noise, on labels that this same boundary produced only pushes those labels further from the boundary, so every mistake becomes more confident and is re-admitted next round.
With no per-class cap and a low threshold ($\tau=0.6$), whichever class the round-0 model favours (class 0, the most confident early on) floods the training set; the prior over predictions shifts toward it, so ambiguous class-2 images (a diagonal cross with noise can resemble a frame) cross the boundary into class 0 and are trained as class 0.
Class 2 is the one that disappears because it started with the fewest and least confident admissions (90 in round 1): it was the class the teacher was least sure about, so it loses the tug-of-war first and, once its pseudo-labels are outnumbered by mislabelled class-2 images, the feedback loop removes it almost completely (2 left in round 4).

### (c) Stopping

If I must use this run's models, I submit the **round-1 model**: it has the highest validation accuracy ($0.83$) and no class has collapsed yet (recalls $0.90, 0.80, 0.80$), while every later round lowers accuracy and class-2 recall. The margin over round 0 is one validation image ($1/30\approx0.033$), so the claim is only "round 1 is the best available", not that self-training clearly helped; the honest report includes that caveat.
The choice must not use the test set because a round chosen by its test score makes the test set a selection set: the reported score would be the best of five noisy evaluations, optimistically biased, and the single locked call would no longer give an unbiased estimate.

### (d) Remedy

Revised recipe: **(ii) + (iii) + (iv) + (vi)**.

* **(ii) Per-class caps that grow across rounds (e.g. 100, 200, 200).** Adopted: the pool is known to be balanced, so equal caps encode that prior and stop one class from flooding the pseudo-labels; growing caps admit only the most confident images early. Risk: if the pool were imbalanced the caps would bias the other way, and a cap does not remove wrong labels inside a class.
* **(iii) Re-initialize the student each round.** Adopted: the student no longer starts on the teacher's boundary, so it fits the labelled + pseudo-labelled set afresh rather than sharpening the teacher's mistakes; it also makes rounds comparable. Risk: costs a full training per round.
* **(iv) Gaussian input noise on the student's training inputs (noisy student).** Adopted: the student cannot memorize the teacher's exact decision on each image, so it learns the noise-stable shape and generalizes beyond the teacher; the teacher still labels clean images. Risk: too much noise underfits; report the noise-only ablation.
* **(vi) Choose the number of rounds on validation accuracy.** Adopted: it is the only label-based signal available and would have stopped this run at round 1. Risk: 30 validation images resolve only about $\pm0.03$, so ties should go to fewer rounds.
* **(i) Raise $\tau$ to 0.99, nothing else.** Rejected: the failure is overconfidence, and by round 3 the admitted images already average $0.98$–$0.99$, so a higher threshold still admits the same confidently wrong class-0 labels while starving class 2 even faster; it treats the symptom, not the feedback loop.
* **(v) Add the 30 validation images to the labelled set.** Rejected: they are the only unbiased data for choosing $\tau$, the caps and the number of rounds; training on them makes their accuracy optimistic and leaves no selection data (the test set must stay locked). Its gain (30 labels) is also small compared with what it destroys.

### (e) Verification without pool labels

1. **Per-class pseudo-label counts each round** (no labels needed): each class at most its cap and the ratio of largest to smallest class count at most $1.5$ in every round (the pool is balanced; in the failed run it was $352/2=176$).
2. **Validation per-class recall each round:** every class at least $0.70$ and none more than $0.10$ below its round-0 value (the failed run had class 2 at $0.20$).
3. **Validation accuracy across rounds:** the chosen round's accuracy at least the round-0 baseline ($0.80$) and no later round lower than the previous by more than one image ($0.033$); additionally, the teacher's accuracy on validation images whose confidence is $\ge\tau$ (pseudo-label each validation image as if it were in the pool, never training on it) at least $0.90$, which estimates the pseudo-label error rate without pool labels.

## Numeric support

In [ ]:
admitted = {1: (180, 150, 90), 2: (260, 210, 40), 3: (330, 240, 12), 4: (352, 246, 2)}
val_acc = {0: 0.80, 1: 0.83, 2: 0.80, 3: 0.73, 4: 0.70}
recall2 = {0: 0.80, 1: 0.80, 2: 0.60, 3: 0.30, 4: 0.20}
pool_per_class = 600 / 3
min_wrong_class0 = admitted[4][0] - pool_per_class
best_round = max(val_acc, key=lambda r: (val_acc[r], -r))
imbalance = {r: max(c) / max(min(c), 1) for r, c in admitted.items()}
print("best round by validation accuracy:", best_round, "| at least", min_wrong_class0, "wrong class-0 pseudo-labels in round 4")
print("max/min class-count ratio per round:", {r: round(v, 1) for r, v in imbalance.items()})

### Answer check

In [ ]:
assert best_round == 1
assert min_wrong_class0 >= 150
assert all(imbalance[r] > 1.5 for r in admitted)                 # every round fails check (1)
assert recall2[4] < 0.70 and val_acc[4] < val_acc[0]              # checks (2) and (3) fail too
assert all(sum(c) <= 600 for c in admitted.values())